In [2]:
from transformers import (
    AutoTokenizer, BertGenerationEncoder, BertGenerationDecoder,
    DataCollatorWithPadding, DataCollatorForSeq2Seq,
    TrainingArguments, Trainer, AutoModelForSeq2SeqLM,
    EncoderDecoderModel
)
import torch
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader

In [4]:
finetune_mlm = False

In [5]:
mlm_chkpt_path = 'entity_masking_mlm/checkpoint-14000'
qa_chkpt_path = 'finetune_qa/checkpoint-2500'

In [9]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
if finetune_mlm:
    encoder = BertGenerationEncoder.from_pretrained(mlm_chkpt_path, bos_token_id=101, eos_token_id=102)
    decoder = BertGenerationDecoder.from_pretrained(
        mlm_chkpt_path, add_cross_attention=True, is_decoder=True, bos_token_id=101, eos_token_id=102
    )
    bert2bert = EncoderDecoderModel(encoder=encoder, decoder=decoder)
else:
    bert2bert = EncoderDecoderModel.from_pretrained(qa_chkpt_path)

In [11]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
df['answers.text'] = pd.Series(df['answers.text'], dtype="string")
df.failurelocation_original = df.failurelocation_original.apply(lambda x: str(list(eval(x))))

In [12]:
df['question'] = 'What are the failure locations?'

In [13]:
df.rename({
    'answers.text': 'context',
    'failurelocation_original': 'answer'
}, axis=1, inplace=True)

In [14]:
df_train = df[df['mode']=='train']
df_val = df[df['mode']=='val']
df_test = df[df['mode']=='test']

In [15]:
ds_train = Dataset.from_pandas(df_train)
ds_val = Dataset.from_pandas(df_val)
ds_test = Dataset.from_pandas(df_test)

In [16]:
def collate_fn(data):
    input_text = [item['context'] for item in data]
    label_text = [item['answer'] for item in data]
    question_text = [item['question'] for item in data]
    tokenized_input = tokenizer(input_text, question_text, return_tensors='pt', truncation=True,
                                max_length=512, padding='max_length')
    label_ids = tokenizer(label_text, return_tensors='pt', truncation=True,
                          max_length=512, padding='max_length')['input_ids']
    return {
        'input_ids': tokenized_input['input_ids'],
        'decoder_input_ids': label_ids,
        'labels': label_ids,
        'attention_mask': tokenized_input['attention_mask']
    }

In [47]:
dl = DataLoader(ds_test, batch_size=8, collate_fn=collate_fn)

In [48]:
item = next(iter(dl))

In [49]:
output = bert2bert(**item)

In [50]:
preds = torch.argmax(output.logits, axis=2)
input_strs = tokenizer.batch_decode(item['input_ids'])
label_strs = tokenizer.batch_decode(item['labels'])
pred_strs = tokenizer.batch_decode(preds)
for i in range(len(item['input_ids'])):
    print("Input:"+"*"*90)
    print(input_strs[i].replace("[PAD]", ""))
    print("Answer:"+"*"*90)
    print(pred_strs[i].replace("[PAD]", ""))
    print("Ground truth:"+"*"*90)
    print(label_strs[i].replace("[PAD]", ""))

Input:******************************************************************************************
[CLS] the equipment accumulator - hydraulic - bladder type, is categorized as fixed asset and has the following boundary : a hydraulic accumulator - bladder type in this database is comprised of : - tank - bladder - oil check valve - gas precharge valve [SEP] what are the failure locations? [SEP]                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
Answer:******************************************************************************************
[CLS] ['gas fill valve ','bladder ','oil valve'] [SEP] 

In [12]:
training_args = TrainingArguments(
    output_dir="finetune_qa",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    remove_unused_columns=False
    
)

trainer = Trainer(
    model=bert2bert,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collate_fn
)

trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/transformers/mo

Epoch,Training Loss,Validation Loss



KeyboardInterrupt

